#### Section B: Question No:2   (10 marks)
Create a convolutional neural network from scratch. Please consider it as a baseline.
Dataset is available under the folder “3_food_classes”.

Conditions to consider:

--Parameters should not cross 20000

--Should not use more than 3 layers (except input and output)

--Use optimizers like  Batch Gradient descent, mini-batch or stochastic


In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint

gpu_devices = tf.config.list_physical_devices('GPU')

if not gpu_devices:
    print("TensorFlow is using the CPU.")
else:
    print(f"TensorFlow is using the following GPU(s): {gpu_devices}")

TensorFlow is using the CPU.


In [10]:
tf.__version__

'2.13.0'

In [2]:
train_dir="3_food_classes/train/"
test_dir="3_food_classes/test/"

In [20]:
datagram = ImageDataGenerator(rescale=1./255)
train_data = datagram.flow_from_directory(
    train_dir,
    target_size=(64, 64),
    batch_size=20,
    # Because these are not binary classification
    class_mode='categorical'
)

test_data = datagram.flow_from_directory(
    test_dir,
    target_size=(64, 64),
    batch_size=20,
    # Also align the final output layer as per this
    class_mode='categorical'
)


Found 225 images belonging to 3 classes.
Found 30 images belonging to 3 classes.


In [15]:
base_model = Sequential([
  Conv2D(32,(3,3), activation='relu', input_shape=(64,64,3)),
  MaxPool2D(2,2),

  Conv2D(64,(3,3), activation='relu'),
  MaxPool2D(2,2),
  
  Flatten(),
  Dense(64, activation='relu'),
  Dense(3, activation='softmax')
])

base_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

base_model.fit(train_data, epochs=10, validation_data=test_data)

Epoch 1/10
12/12 [==============================] - 4s 323ms/step - loss: 1.1396 - accuracy: 0.3511 - val_loss: 1.0631 - val_accuracy: 0.4667
Epoch 2/10
12/12 [==============================] - 1s 118ms/step - loss: 0.9398 - accuracy: 0.5911 - val_loss: 1.0466 - val_accuracy: 0.4667
Epoch 3/10
12/12 [==============================] - 1s 115ms/step - loss: 0.9220 - accuracy: 0.5822 - val_loss: 1.0611 - val_accuracy: 0.5333
Epoch 4/10
12/12 [==============================] - 1s 113ms/step - loss: 0.7722 - accuracy: 0.6711 - val_loss: 1.1443 - val_accuracy: 0.4667
Epoch 5/10
12/12 [==============================] - 1s 112ms/step - loss: 0.6703 - accuracy: 0.7289 - val_loss: 1.2171 - val_accuracy: 0.4333
Epoch 6/10
12/12 [==============================] - 1s 114ms/step - loss: 0.5754 - accuracy: 0.7867 - val_loss: 1.2195 - val_accuracy: 0.4667
Epoch 7/10
12/12 [==============================] - 1s 113ms/step - loss: 0.4861 - accuracy: 0.8267 - val_loss: 1.2675 - val_accuracy: 0.4667
Epoch 

#### Section B: Question No:3   (20 marks)
Improve the baseline model performance and save the weights of improved model

Conditions to consider:

--Apply Data Augmentation

--No parameter limit

--Can use more than 3 (except input and output)

--Use any optimizers of your choice 

--Use callbacks to save the best model weights




In [25]:
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

# Data augmentation
train_datagen_aug = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=90,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Go for larger image size for better performance
train_data_aug = train_datagen_aug.flow_from_directory(
    train_dir, target_size=(32, 32), batch_size=32, class_mode='categorical')

test_data = train_datagen_aug.flow_from_directory(
    test_dir, target_size=(32, 32), batch_size=16, class_mode='categorical')

# Improved model with more layers / No parameter Limit
improved_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Second convolutional layer
    Conv2D(16, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Third convolutional layer
    Conv2D(8, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Fully connected layers
    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3 classes for food classification
])

# Compile the improved model
improved_model.compile(optimizer='adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

# Set up a checkpoint/callback to save the best weights
checkpoint = ModelCheckpoint("best_model.h5", save_best_only=True, monitor='val_accuracy', mode='max')

# Train the improved model
improved_model.fit(train_data_aug, validation_data=test_data, epochs=20, callbacks=[checkpoint])


Found 225 images belonging to 3 classes.


Found 30 images belonging to 3 classes.
Epoch 1/20
8/8 [==============================] - 3s 203ms/step - loss: 1.7873 - accuracy: 0.3867 - val_loss: 1.1011 - val_accuracy: 0.3000
Epoch 2/20


/home/vishrut/.pyenv/versions/scrape/lib/python3.10/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


8/8 [==============================] - 1s 174ms/step - loss: 1.9428 - accuracy: 0.3689 - val_loss: 1.1045 - val_accuracy: 0.3333
Epoch 3/20
8/8 [==============================] - 1s 163ms/step - loss: 1.4220 - accuracy: 0.4578 - val_loss: 1.1053 - val_accuracy: 0.3000
Epoch 4/20
8/8 [==============================] - 1s 170ms/step - loss: 1.5351 - accuracy: 0.4400 - val_loss: 1.1047 - val_accuracy: 0.3667
Epoch 5/20
8/8 [==============================] - 1s 181ms/step - loss: 1.4645 - accuracy: 0.4756 - val_loss: 1.1239 - val_accuracy: 0.2000
Epoch 6/20
8/8 [==============================] - 1s 158ms/step - loss: 1.5649 - accuracy: 0.3867 - val_loss: 1.1097 - val_accuracy: 0.3667
Epoch 7/20
8/8 [==============================] - 1s 160ms/step - loss: 1.4513 - accuracy: 0.4800 - val_loss: 1.1105 - val_accuracy: 0.3333
Epoch 8/20
8/8 [==============================] - 1s 160ms/step - loss: 1.2983 - accuracy: 0.4844 - val_loss: 1.1052 - val_accuracy: 0.3667
Epoch 9/20
8/8 [===============